# Lesson 12 - 使用代理便條本減少聊天記錄

此筆記本展示如何使用 Microsoft 代理框架管理長對話中的上下文。隨著對話增長，標記數量增加 — 最終超出模型的上下文視窗。我們通過**上下文摘要模式**和持久記憶的**代理便條本**來解決這個問題。

## 你將學習到：
1. **為什麼上下文管理很重要**：了解標記限制和上下文視窗
2. **上下文感知代理**：構建管理自身對話上下文的代理
3. **上下文摘要模式**：使用工具濃縮對話歷史
4. **代理便條本**：可在上下文縮減後保留的持久記憶

## 先決條件：
- 已配置環境變數的 Azure OpenAI 設置
- 了解之前課程中基本的代理概念


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## 為何上下文管理很重要

每個大型語言模型（LLM）都有有限的**上下文視窗**——即它在單次請求中能處理的最大標記數。隨著多輪對話的進行：

- **標記數量隨每條使用者訊息與助理回覆線性成長**。
- **提示標記佔主導成本**，因為整個對話歷史每輪都會重新傳送。
- 最終對話會**超過上下文視窗**，模型要麼被截斷，要麼報錯。

### 上下文管理策略

| 策略 | 運作方式 | 取捨 |
|---|---|---|
| **截斷** | 丟棄最舊的訊息 | 失去早期上下文 |
| **摘要** | 將較舊訊息濃縮成摘要 | 失去部分細節，但保留重點 |
| **草稿簿 / 外部記憶** | 將關鍵事實存放在對話外部 | 需要調用工具，但可承受任何縮減 |

在本筆記本中，我們結合**摘要**與**草稿簿工具**，讓代理即使在對話歷史被濃縮時也能維持連貫性。


## 創建一個具備上下文感知的代理


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## 模擬一段長時間對話

讓我們透過多輪對話來看看上下文是如何累積的。代理應該在多輪對話中保留關鍵細節（偏好、預算、旅行日期），並展示連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

請注意代理人如何保留早期回合的上下文——它知道日本、壽司、寺廟、攝影、3000美元預算、獨自旅行及四月的時間範圍。在短暫的對話中這運作良好，但隨著對話增長，重新傳送完整歷史會變得昂貴。

讓我們繼續對話，增加更多回合以觀察上下文累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話的增加，我們可以使用**摘要工具**將累積的上下文濃縮成一個簡潔的格式。代理會調用這個工具來記錄關鍵偏好，以便即使較舊的訊息被刪除，重要資訊仍然得以保留。

這個模式是更複雜歷史縮減的基石：
1. 代理從對話中識別關鍵事實
2. 它調用摘要工具來保存這些資訊
3. 較舊的訊息可以安全移除，因為摘要已捕捉到重要內容

以下我們定義了一個 `summarize_preferences` 工具，代理可以調用它來記錄所學到的簡潔摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Summary

在本課程中，您學會了如何使用 Microsoft Agent Framework 來管理長時間運行的代理對話中的上下文：

### 主要概念
- **上下文視窗是有限的** — 對話歷史中的每個標記都會產生費用並計入限制。
- **摘要工具** 讓代理能將累積的上下文濃縮成簡潔的摘要，減少標記使用量，同時保留重要資訊。
- **代理草稿本** 提供持久的外部記憶，可在任何對話縮減後繼續存在。

### 您所建立的內容
- 一個 **具上下文感知的代理**，能在多輪對話中保持連貫性
- 一個 **摘要工具** (`summarize_preferences`)，以簡潔格式記錄重要的用戶細節
- 一個 **多輪對話**，展示上下文保持與變更處理

### 實際應用
- **客戶服務機器人**：在長時間的支援會話中記住偏好設定
- **個人助理**：追蹤正在進行的專案，無需重複說明上下文
- **教育輔導員**：在多次互動中保持學生進度

### 下一步
- 實作具備檔案持久化的完整草稿本工具
- 在摘要後加入自動歷史記錄截斷功能
- 結合向量資料庫進行語意記憶搜尋
- 建立能在數天後以完整上下文重新開始對話的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件係使用人工智能翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 所翻譯。儘管我們致力於準確性，但請注意，機器翻譯可能包含錯誤或不準確之處。原始語言文件應被視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。本公司不對因使用本翻譯而引起的任何誤解或誤釋負責。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
